<a href="https://colab.research.google.com/github/SergiSama/UIC-CRB1-2026-2027/blob/main/m3_derivatives_gradients_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 3 · Notebook 1 — Derivatives, Gradients & Optimisation
### Computing, Robotics & Bionics · companion to A. Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch*

A **self-contained** differential-calculus reference (no other notebook needed). It covers Géron's
`math_differential_calculus` and autodiff material, on biomedical examples: the derivative and chain rule,
numerical derivatives and gradient checking, the gradient and the tangent plane, the total differential
along a path, Taylor expansions, the Jacobian (the robot's Jacobian!) as a best linear approximation, the three losses you will
differentiate, backpropagation and one layer's gradients, why depth makes gradients vanish or explode, why backprop runs backwards,
**gradient descent from scratch** with the learning-rate law read off the Hessian, and the same gradient computed three ways — by hand, by finite
difference, and by **PyTorch autograd**.

Run in **Google Colab** (*Runtime → Run all*). The core is offline; the autograd cells use PyTorch
(pre-installed in Colab) and are guarded so the notebook never breaks.

**Contents**
1. The derivative and the chain rule
2. Numerical derivatives & gradient checking
3. The gradient, steepest ascent and the tangent plane
4. The total differential: a path through a field
5. Taylor: the local model optimisation actually minimises
6. The Jacobian (the robot's Jacobian)
7. The three losses you will actually differentiate
8. Backpropagation on a tiny graph, and one layer's gradients
9. Why depth makes gradients vanish or explode — and why backprop runs backwards
10. Gradient descent from scratch, and the learning-rate law
11. The same gradient three ways (autograd)

Every section ends with an **Exercise**; run the **Solution** cell to check.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
print("ready")

---
## 1 · The derivative and the chain rule

The derivative is the slope of the tangent — the rate of change. We plot a function and its tangent line
at a point, using the analytic derivative.

In [ ]:
f  = lambda x: x**2
df = lambda x: 2*x                  # analytic derivative
x0 = 1.5
xs = np.linspace(-1, 3, 200)
tangent = f(x0) + df(x0)*(xs - x0)  # y = f(x0) + f'(x0)(x - x0)
fig, ax = plt.subplots(figsize=(6,4))
ax.plot(xs, f(xs), label="f(x)=x^2")
ax.plot(xs, tangent, "--", label=f"tangent at x={x0} (slope {df(x0)})")
ax.scatter([x0],[f(x0)], color="crimson", zorder=5)
ax.legend(); ax.grid(alpha=0.3); ax.set_title("Derivative = slope of the tangent"); plt.show()

In [ ]:
# Chain rule in action: derivative of the squared-error loss L(w) = 0.5*(w*x - y)^2
x, y = 3.0, 7.0
L  = lambda w: 0.5*(w*x - y)**2
dL = lambda w: (w*x - y)*x          # chain rule: error * input
print("L(2) =", L(2.0), " dL/dw at w=2 =", dL(2.0))

**Exercise 1.** A drug's plasma concentration follows first-order elimination `C(t) = C0 * np.exp(-k*t)`, with `C0 = 50.0` mg/L and `k = 0.35` per hour. Using the chain rule, find the analytic derivative `dC/dt` (the instantaneous elimination rate) as a function of `t`, evaluate it at `t = 2` hours, and confirm the sign shows the concentration is decreasing.

> 🤖 *Gemini tip:* "Given an exponential decay function C(t) = C0 * e^(-kt), walk me through applying the chain rule to find dC/dt, then show me how to evaluate both C(t) and its derivative in NumPy at a specific time."

In [ ]:
C0, k = 50.0, 0.35

# Your code here

---
## 2 · Numerical derivatives & gradient checking

When you only have *code* for `f`, estimate the slope by finite differences. The **central** difference is
much more accurate than the forward one. Comparing an analytic gradient to a central difference —
**gradient checking** — catches bugs.

In [ ]:
def forward_diff(f, x, h=1e-5):  return (f(x+h) - f(x)) / h
def central_diff(f, x, h=1e-5):  return (f(x+h) - f(x-h)) / (2*h)

x0 = 1.5
exact = df(x0)
print("exact        :", exact)
print("forward diff :", forward_diff(f, x0), " err", abs(forward_diff(f,x0)-exact))
print("central diff :", central_diff(f, x0), " err", abs(central_diff(f,x0)-exact))

In [ ]:
# Gradient check on the loss gradient
w = 2.0
print("analytic dL/dw:", dL(w))
print("numeric  dL/dw:", central_diff(L, w))
print("agree:", np.isclose(dL(w), central_diff(L, w), atol=1e-6))

**Exercise 2.** A respiration belt reports chest expansion as `np.sin` of the breathing phase, so
its derivative is the instantaneous airflow. Estimate the derivative of `np.sin` at `x = 1.0` rad
with the central difference and compare to the exact value `cos(1.0)`.

> 🤖 *Gemini tip:* "Show me how to implement the central-difference formula for a numerical derivative in NumPy, and how to compare it to an exact analytic derivative."


In [ ]:
# Your code here

---
## 3 · The gradient, steepest ascent and the tangent plane

For a function of several variables the **gradient** collects the partial derivatives into a
vector. It points in the direction of steepest increase, its length is the steepest rate, and it
is perpendicular to the contours. We draw the gradient field over the contours of
`f(x,y)=x^2+3xy+y^2`, then prove the steepest-ascent claim by sweeping directions, and finally
connect the 2-D gradient to the 3-D tangent plane.


In [ ]:
def fxy(x, y):  return x**2 + 3*x*y + y**2
def grad_fxy(x, y):  return np.array([2*x + 3*y, 3*x + 2*y])   # [df/dx, df/dy]
print("grad at (1,2):", grad_fxy(1, 2))

gx, gy = np.meshgrid(np.linspace(-3,3,30), np.linspace(-3,3,30))
fig, ax = plt.subplots(figsize=(6,5))
ax.contour(gx, gy, fxy(gx, gy), levels=20, cmap="viridis")
U, V = grad_fxy(gx, gy)
ax.quiver(gx, gy, U, V, color="crimson", alpha=0.6)
ax.set_title("Gradient field points uphill, perpendicular to contours")
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_aspect("equal"); plt.show()

### 3.1 · Why the gradient *is* the steepest direction

Pick a unit direction $\mathbf{u}$ and walk that way. The rate at which $f$ changes is the
**directional derivative**

$$D_{\mathbf u} f(\mathbf p) \;=\; \lim_{h\to 0}\frac{f(\mathbf p + h\mathbf u)-f(\mathbf p)}{h}\;=\;\nabla f(\mathbf p)\cdot\mathbf u
 \;=\;\lVert\nabla f\rVert\,\lVert\mathbf u\rVert\cos\theta\;=\;\lVert\nabla f\rVert\cos\theta .$$

Everything except $\cos\theta$ is fixed once you stand at $\mathbf p$, so the rate is largest when
$\cos\theta = 1$ — when $\mathbf u$ points **along** $\nabla f$ — and it is zero when
$\theta = 90^\circ$, i.e. along the contour. That single line is the whole reason gradient descent
steps in $-\nabla f$: it is not *a* downhill direction, it is the *steepest* one.


In [ ]:
# Sweep every direction at one point of the concentration field and read off the best one.
p = np.array([1.0, 0.5])                       # a point in tissue, mm
g = grad_fxy(*p)

angles = np.linspace(0, 2*np.pi, 721)
U  = np.stack([np.cos(angles), np.sin(angles)])   # 2 x 721 unit directions
Du = g @ U                                        # directional derivative in each one

print("gradient at p            :", g, "   |grad| =", round(np.linalg.norm(g), 4))
print("best rate over directions:", round(Du.max(), 4),
      "at", round(np.degrees(angles[Du.argmax()]), 1), "deg")
print("gradient's own direction :", round(np.degrees(np.arctan2(g[1], g[0])), 1), "deg")

# and the same directional derivative straight from the definition, numerically
h = 1e-6
for name, u in [("along grad", g/np.linalg.norm(g)),
                ("along contour", np.array([-g[1], g[0]])/np.linalg.norm(g))]:
    numeric = (fxy(*(p + h*u)) - fxy(*(p - h*u))) / (2*h)
    print(f"{name:14s}: grad.u = {g@u: .4f}   finite difference = {numeric: .4f}")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(np.degrees(angles), Du, color="#0E7C7B")
ax.axhline(0, color="gray", lw=0.8)
ax.axvline(np.degrees(np.arctan2(g[1], g[0])) % 360, color="crimson", ls="--",
           label="direction of grad f")
ax.axhline(np.linalg.norm(g), color="crimson", ls=":", label="|grad f|")
ax.set_xlabel("direction u  (degrees)"); ax.set_ylabel("rate of change  grad f . u")
ax.set_title("The rate of change is a cosine: biggest along the gradient")
ax.legend(); ax.grid(alpha=0.3); plt.show()


### 3.2 · The 2-D gradient and the 3-D tangent plane

This is the point that trips everyone up, so it is worth doing slowly. The gradient of
$f(x,y)$ lives in the **2-D input plane** — it has two components and it is drawn *on the map*,
not on the hill. The graph of $f$ is a **surface in 3-D**, $z = f(x,y)$, and at a point it has a
**tangent plane**

$$z \;=\; f(x_0,y_0) + f_x\,(x-x_0) + f_y\,(y-y_0)
\qquad\Longleftrightarrow\qquad
f_x\,x + f_y\,y - z \;=\; \text{const},$$

whose normal vector is therefore $\mathbf n = (f_x,\,f_y,\,-1)$ — the 2-D gradient with a $-1$
stapled on. The two objects are the same theorem one dimension apart: write
$F(x,y,z) = f(x,y) - z$, so the surface is the level set $F = 0$; then
$\nabla F = (f_x, f_y, -1)$, and a gradient is always perpendicular to its own level set. The
$-1$ is not a convention, it is $\partial F/\partial z$. The picture below is drawn with the same
visual scale on all three axes, so the normal really does look perpendicular.


In [ ]:
# One point, three objects: the 2-D gradient, the tangent plane, and the plane's normal.
# A gentle surface, so the picture can be drawn to scale and the normal really looks normal:
# read it as a skin-temperature map T(x, y) over a patch of forearm, in degC above baseline.
def Tmap(x, y):       return 0.5*x**2 + 0.3*x*y + 0.4*y**2
def grad_Tmap(x, y):  return np.array([x + 0.3*y, 0.3*x + 0.8*y])

x0, y0 = 1.0, 0.5
fx, fy = grad_Tmap(x0, y0)
z0 = Tmap(x0, y0)
n  = np.array([fx, fy, -1.0])                      # normal to the tangent plane
print("2-D gradient (drawn on the map):", np.array([fx, fy]).round(4))
print("3-D normal   (to the plane)    :", n.round(4))

# every direction lying IN the tangent plane is perpendicular to n
for du, dv in [(1.0, 0.0), (0.0, 1.0), (0.7, -0.3)]:
    in_plane = np.array([du, dv, fx*du + fy*dv])   # rise = gradient . step
    print(f"  step ({du:+.1f},{dv:+.1f}) -> in-plane vector {in_plane.round(3)}"
          f"   n . v = {n @ in_plane:+.1e}")

LO, HI = -0.4, 2.4
gx3, gy3 = np.meshgrid(np.linspace(LO, HI, 60), np.linspace(LO - 0.7, HI - 0.7, 60))
plane = z0 + fx*(gx3 - x0) + fy*(gy3 - y0)
ZLO, ZHI = -1.4, 2.8
surf = np.where(Tmap(gx3, gy3) > ZHI, np.nan, Tmap(gx3, gy3))   # keep the drawing inside the box
plane = np.where((plane > ZHI) | (plane < ZLO), np.nan, plane)

fig = plt.figure(figsize=(7.0, 5.4))
ax = fig.add_subplot(111, projection="3d")
ax.plot_surface(gx3, gy3, surf,  alpha=0.55, cmap="viridis", linewidth=0, rstride=2, cstride=2)
ax.plot_surface(gx3, gy3, plane, alpha=0.40, color="#C77700", linewidth=0, rstride=2, cstride=2)
ax.scatter([x0], [y0], [z0], color="crimson", s=45, depthshade=False)
ax.quiver(x0, y0, z0, *n, color="crimson", lw=2.2, arrow_length_ratio=0.15)
ax.text(x0 + 0.95*n[0], y0 + 0.95*n[1], z0 + 0.95*n[2] - 0.25, "n", color="crimson", fontsize=12)
ax.quiver(x0, y0, z0, fx, fy, 0, color="navy", lw=2.2, arrow_length_ratio=0.15)
ax.text(x0 + 0.55*fx, y0 + 0.55*fy, z0 + 0.32, "grad T", color="navy", fontsize=11)
ax.set_xlim(LO, HI); ax.set_ylim(LO - 0.7, HI - 0.7); ax.set_zlim(ZLO, ZHI)
ax.set_xlabel("x (cm)", labelpad=-4); ax.set_ylabel("y (cm)", labelpad=-4)
ax.set_zlabel("z = T(x,y)", labelpad=-4)
ax.set_xticks([0, 1, 2]); ax.set_yticks([0, 1]); ax.set_zticks([-1, 0, 1, 2])
ax.tick_params(labelsize=8, pad=0)
ax.set_box_aspect((HI - LO, HI - LO, ZHI - ZLO), zoom=0.80)   # one visual unit per axis unit
ax.view_init(elev=20, azim=-62)
ax.set_title("n = (T_x, T_y, -1) is perpendicular to the tangent plane", pad=-2)
plt.subplots_adjust(left=0.02, right=0.98, top=0.98, bottom=0.02); plt.show()


**Exercise 3.** Read `f(x,y) = x**2 + 3*x*y + y**2` above as a concentration field around an injection site, in tissue coordinates `(x, y)` measured in mm. Compute the gradient at the point `(x, y) = (-1, 2)` and report it as a **unit vector** giving the direction of fastest local increase in concentration.

> 🤖 *Gemini tip:* "Given a Python function of two variables and its gradient function, show me how to evaluate the gradient at a specific point and turn it into a unit vector giving the direction of steepest increase."

In [ ]:
p0 = (-1.0, 2.0)

# Your code here

**Exercise 4.** A bolus injected into tissue diffuses into a Gaussian blob of concentration
`C(x, y) = Cmax * np.exp(-(x**2 + y**2) / (2*sigma**2))`, with `Cmax = 120.0` mg/L and
`sigma = 1.5` mm, measured from the injection site. At the sampling point `(x, y) = (1.0, -0.8)` mm:
(a) derive `grad C` by hand and evaluate it; (b) confirm it is the steepest direction by sweeping
360 unit directions and checking that none beats `|grad C|`; (c) write down the normal vector of the
tangent plane to the surface `z = C(x, y)` at that point.

> 🤖 *Gemini tip:* "For a 2-D Gaussian f(x,y) = A*exp(-(x^2+y^2)/(2s^2)), derive the partial derivatives by the chain rule, then show me NumPy code that evaluates the directional derivative grad f . u over many unit directions u and checks the maximum equals |grad f|."


In [ ]:
Cmax, sigma = 120.0, 1.5
q = np.array([1.0, -0.8])

# Your code here

---
## 4 · The total differential: a path through a field

Physics writes the chain rule in a shape worth naming. A parameter $t$ picks a **point**
$\mathbf r(t) = (x(t), y(t), z(t))^{\mathsf T}$, and a **scalar field** $V$ assigns a number to every
point. Compose them and you get a plain function of one variable, $t \mapsto V(\mathbf r(t))$. The
chain rule through the vector middle step gives the **total differential**

$$\mathrm dV \;=\; \frac{\partial V}{\partial x}\mathrm dx + \frac{\partial V}{\partial y}\mathrm dy
   + \frac{\partial V}{\partial z}\mathrm dz \;=\; \nabla V \cdot \mathrm d\mathbf r,
\qquad\text{and along the path}\qquad
\frac{\mathrm dV}{\mathrm dt} \;=\; \nabla V \cdot \dot{\mathbf r}(t).$$

Read it as a dot product and two facts fall out for free: you climb fastest when the velocity lines
up with the gradient, and **moving along a level surface costs nothing** — $\dot{\mathbf r}
\perp \nabla V$ makes $\mathrm dV/\mathrm dt$ exactly zero. This is also the shape backpropagation
uses: one scalar at the end, many parameters at the start, one dot product per step.


In [ ]:
# A catheter tip advancing through the drug cloud left by an injection.
C0c, sig_mm = 200.0, 1.2                           # mg/L at the injection site, mm spread

def Cfield3(r):                                    # scalar field: concentration at a point
    return C0c*np.exp(-(r @ r)/(2*sig_mm**2))

def gradC3(r):                                     # chain rule: grad C = C * (-r / sigma^2)
    return Cfield3(r) * (-r)/sig_mm**2

def path(t):                                       # helical advance of the catheter tip, mm
    return np.array([0.8*np.cos(t), 0.8*np.sin(t), 0.3*t])

def vel(t):                                        # r'(t), mm per second
    return np.array([-0.8*np.sin(t), 0.8*np.cos(t), 0.3])

t = 1.1
r, v = path(t), vel(t)
analytic = gradC3(r) @ v                           # dC/dt = grad C . r'
h = 1e-6
numeric  = (Cfield3(path(t+h)) - Cfield3(path(t-h))) / (2*h)
print("position r(t)      :", r.round(4))
print("velocity r'(t)     :", v.round(4))
print("grad C at r(t)     :", gradC3(r).round(4))
print("dC/dt = grad C . v :", round(analytic, 6))
print("dC/dt numerically  :", round(numeric, 6))
print("agree:", np.isclose(analytic, numeric, rtol=1e-6))


In [ ]:
# Two paths through the same field: one that climbs, one that stays on a level surface.
def flat(t):     return np.array([0.8*np.cos(t), 0.8*np.sin(t), 0.3])   # constant radius
def flat_vel(t): return np.array([-0.8*np.sin(t), 0.8*np.cos(t), 0.0])

ts = np.linspace(0, 6, 300)
C_helix = np.array([Cfield3(path(s))   for s in ts])
C_flat  = np.array([Cfield3(flat(s))   for s in ts])
dC_helix = np.array([gradC3(path(s)) @ vel(s)      for s in ts])
dC_flat  = np.array([gradC3(flat(s)) @ flat_vel(s) for s in ts])

print("helical path: dC/dt runs over",
      f"[{dC_helix.min():.2f}, {dC_helix.max():.2f}] mg/L/s - always negative once the tip",
      "starts advancing, because the velocity keeps a component along -grad C")
print("level path  : max |dC/dt| =", f"{np.abs(dC_flat).max():.2e}",
      " (grad C is perpendicular to the velocity everywhere)")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
a1.plot(ts, C_helix, color="#0E7C7B", label="helix (advancing)")
a1.plot(ts, C_flat,  color="#C77700", label="circle at constant depth")
a1.set_xlabel("t (s)"); a1.set_ylabel("C along the path (mg/L)")
a1.set_title("What the sensor on the tip reads"); a1.legend(); a1.grid(alpha=0.3)
a2.plot(ts, dC_helix, color="#0E7C7B", label="helix")
a2.plot(ts, dC_flat,  color="#C77700", label="level path: dC/dt = 0")
a2.axhline(0, color="gray", lw=0.8)
a2.set_xlabel("t (s)"); a2.set_ylabel("dC/dt = grad C . r'  (mg/L/s)")
a2.set_title("The total differential along each path"); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()


**Exercise 5.** A wrist-worn IMU tracks a fingertip sweeping through a tissue-oxygenation field
`S(x, y) = 95.0 - 6.0*np.exp(-((x-1.0)**2 + (y+0.5)**2)/2.0)` (percent saturation, position in cm).
The fingertip follows `r(t) = (1.5*np.cos(t), 1.5*np.sin(t))`. Compute `dS/dt` at `t = 0.7` s two
ways — analytically as `grad S . r'` and numerically by central-differencing `S(r(t))` — and confirm
they agree. Then say, in one line, what it would mean physically for `dS/dt` to be zero.

> 🤖 *Gemini tip:* "I have a scalar field S(x,y) and a parametric path r(t) in NumPy. Show me how to compute dS/dt using the chain rule grad S . r'(t), and how to verify it with a central finite difference of the composed function S(r(t))."


In [ ]:
Spath = lambda t: np.array([1.5*np.cos(t), 1.5*np.sin(t)])
t_q = 0.7

# Your code here

---
## 5 · Taylor: the local model optimisation actually minimises

Every optimiser works on a **local polynomial model** of the loss, not the loss itself:

$$f(x_0+h) \;=\; f(x_0) + f'(x_0)\,h + \tfrac12 f''(x_0)\,h^2 + O(h^3),
\qquad
f(\mathbf x_0 + \mathbf p) \;\approx\; f(\mathbf x_0) + \nabla f^{\mathsf T}\mathbf p
 + \tfrac12\,\mathbf p^{\mathsf T}\mathbf H\,\mathbf p .$$

Truncate after the linear term and minimising the model *is* gradient descent; keep the quadratic
term and it is Newton's method. The same expansion explains the error orders you measured in
Section 2, and it is where the $O(h)$ and $O(h^2)$ labels came from.


In [ ]:
# Orders 0, 1, 2 of the drug-elimination curve around t0 = 2 h.
C0t, kt = 50.0, 0.35
Ct  = lambda t: C0t*np.exp(-kt*t)
t0t = 2.0
d0, d1, d2 = Ct(t0t), -kt*Ct(t0t), kt**2*Ct(t0t)     # C, C', C'' at t0

ts = np.linspace(0, 6, 300)
T0 = d0*np.ones_like(ts)
T1 = d0 + d1*(ts - t0t)
T2 = T1 + 0.5*d2*(ts - t0t)**2

for dt in [0.1, 0.5, 2.0]:
    true = Ct(t0t + dt)
    print(f"dt={dt:4.1f}  exact {true:7.3f} | const {d0:7.3f} | linear "
          f"{d0+d1*dt:7.3f} | quadratic {d0+d1*dt+0.5*d2*dt**2:7.3f}")

fig, ax = plt.subplots(figsize=(6.6, 4))
ax.plot(ts, Ct(ts), color="black", lw=2, label="C(t) = C0 e^(-kt)")
ax.plot(ts, T0, ":",  color="#B23A48", label="order 0 (value)")
ax.plot(ts, T1, "--", color="#C77700", label="order 1 (tangent)")
ax.plot(ts, T2, "-.", color="#0E7C7B", label="order 2 (curvature)")
ax.scatter([t0t], [d0], color="crimson", zorder=5)
ax.set_ylim(0, 60); ax.set_xlabel("t (h)"); ax.set_ylabel("concentration (mg/L)")
ax.set_title("Taylor: each extra term buys you a wider neighbourhood")
ax.legend(); ax.grid(alpha=0.3); plt.show()


In [ ]:
# Taylor also predicts the error of the two finite differences from Section 2.
hs  = np.logspace(-8, -1, 40)
fwd = np.abs((Ct(t0t + hs) - Ct(t0t))/hs - d1)                 # truncation O(h)
cen = np.abs((Ct(t0t + hs) - Ct(t0t - hs))/(2*hs) - d1)        # truncation O(h^2)

band = (hs > 1e-4) & (hs < 1e-1)     # below this, floating-point round-off takes over
sf = np.polyfit(np.log10(hs[band]), np.log10(fwd[band]), 1)[0]
sc = np.polyfit(np.log10(hs[band]), np.log10(cen[band]), 1)[0]
print(f"measured slope, forward difference: {sf:.2f}   (Taylor predicts 1)")
print(f"measured slope, central difference: {sc:.2f}   (Taylor predicts 2)")
print(f"best central h here: {hs[cen.argmin()]:.1e}  with error {cen.min():.1e}")

fig, ax = plt.subplots(figsize=(6.2, 4))
ax.loglog(hs, fwd, "o-", ms=3, color="#C77700", label=f"forward  (slope {sf:.2f})")
ax.loglog(hs, cen, "s-", ms=3, color="#0E7C7B", label=f"central  (slope {sc:.2f})")
ax.set_xlabel("step h"); ax.set_ylabel("absolute error in C'(t0)")
ax.set_title("Truncation falls with h; round-off rises as h shrinks")
ax.legend(); ax.grid(alpha=0.3, which="both"); plt.show()


In [ ]:
# Multivariate Taylor: for a quadratic loss the 2nd-order model is EXACT,
# so one Newton step lands on the minimum. Least squares on two VO2 predictors.
rng = np.random.default_rng(0)
m = 60
X = np.c_[np.ones(m), rng.normal(size=m), rng.normal(size=m)]   # bias, HR, workload
w_true = np.array([8.0, 2.5, -1.2])
yv = X @ w_true + rng.normal(scale=0.4, size=m)                 # VO2, mL/kg/min

loss = lambda w: 0.5*np.sum((X @ w - yv)**2)
grad = lambda w: X.T @ (X @ w - yv)
H    = X.T @ X                                                  # Hessian: constant here

w0 = np.zeros(3)
p  = np.linalg.solve(H, -grad(w0))          # minimiser of the quadratic model
w_newton = w0 + p
w_star   = np.linalg.lstsq(X, yv, rcond=None)[0]
print("one Newton step :", w_newton.round(4))
print("least squares   :", w_star.round(4))
print("distance        :", f"{np.linalg.norm(w_newton - w_star):.2e}", "-> the model was exact")

# and the model really does reproduce the loss, not just its minimum
for s in [0.05, 0.2, 1.0]:
    q = s*p
    model = loss(w0) + grad(w0) @ q + 0.5*q @ H @ q
    print(f"  step {s:4.2f}*p : true loss {loss(w0+q):10.4f} | 2nd-order model {model:10.4f}")


**Exercise 6.** The logistic sigmoid `sig(z) = 1/(1 + np.exp(-z))` is the activation behind every
binary classifier in this course (malignant vs benign, for instance). Build its second-order Taylor
expansion about `z0 = 0`, where `sig(0) = 0.5`, `sig'(0) = 0.25` and `sig''(0) = 0`. Plot the sigmoid
against the linear model on `z` in `[-4, 4]`, and report the largest `|z|` for which the linear model
stays within 0.02 of the true sigmoid. What does the vanishing second derivative tell you about the
shape of the sigmoid at the origin?

> 🤖 *Gemini tip:* "Derive the first and second derivatives of the logistic sigmoid, evaluate them at z=0, and show me NumPy code that plots the function against its first-order Taylor approximation and finds the range where the approximation error stays under a tolerance."


In [ ]:
sig = lambda z: 1/(1 + np.exp(-z))
zs  = np.linspace(-4, 4, 801)

# Your code here

---
## 6 · The Jacobian (the robot's Jacobian)

For a vector-valued function the **Jacobian** is the matrix of all partial derivatives. Here it is
the 2-link arm's forward kinematics — exactly the manipulator Jacobian from the Robotics & Bionics
unit — and we confirm the analytic Jacobian against a numerical one.


In [ ]:
l1, l2 = 0.3, 0.3
def fk(q):                          # forward kinematics: joints -> (x, y)
    t1, t2 = q
    return np.array([l1*np.cos(t1) + l2*np.cos(t1+t2),
                     l1*np.sin(t1) + l2*np.sin(t1+t2)])

def jacobian_analytic(q):
    t1, t2 = q
    return np.array([
        [-l1*np.sin(t1) - l2*np.sin(t1+t2), -l2*np.sin(t1+t2)],
        [ l1*np.cos(t1) + l2*np.cos(t1+t2),  l2*np.cos(t1+t2)],
    ])

def jacobian_numeric(fn, q, h=1e-6):       # central difference, column by column
    q = np.asarray(q, float); m = len(fn(q)); J = np.zeros((m, len(q)))
    for j in range(len(q)):
        dq = np.zeros_like(q); dq[j] = h
        J[:, j] = (fn(q+dq) - fn(q-dq)) / (2*h)
    return J

q = np.array([np.radians(30), np.radians(45)])
print("analytic J:\n", jacobian_analytic(q))
print("numeric  J:\n", jacobian_numeric(fk, q))
print("match:", np.allclose(jacobian_analytic(q), jacobian_numeric(fk, q)))

**Why "best linear approximation"?** The phrase is a claim about how fast the error vanishes. Take a
small step of length $h$ in some direction $\mathbf u$. The *constant* model $f(\mathbf q)$ is off by
$O(h)$ — it ignores the motion entirely. The *linear* model $f(\mathbf q)+\mathbf J\,(h\mathbf u)$ is off by
only $O(h^2)$: halve the step and the error **quarters**. That quadratic shrinking is what makes $\mathbf J$
the best linear approximation, and it is the same Taylor bound (Section 5) that set the error of the
finite differences in Section 2.

In [ ]:
# Step away from q along a fixed direction and compare the two local models of the arm.
u_lin = np.array([1.0, -0.6]) / np.linalg.norm([1.0, -0.6])      # a joint-space direction
J_q = jacobian_analytic(q)
hs_lin = np.array([0.2, 0.1, 0.05, 0.025, 0.0125])
err_const = np.array([np.linalg.norm(fk(q + h*u_lin) - fk(q)) for h in hs_lin])
err_lin = np.array([np.linalg.norm(fk(q + h*u_lin) - fk(q) - J_q @ (h*u_lin)) for h in hs_lin])

print(" step h   | constant model: err, err/h  | linear model: err, err/h^2")
for h, e0, e1 in zip(hs_lin, err_const, err_lin):
    print(f" {h:7.4f}  |   {e0:.3e}  {e0/h:.4f}     |   {e1:.3e}  {e1/h**2:.4f}")
print("err/h is constant for the constant model (error ~ h);")
print("err/h^2 is constant for the linear model (error ~ h^2): halving h quarters the error.")

fig, ax = plt.subplots(figsize=(5.8, 3.8))
ax.loglog(hs_lin, err_const, "o-", color="#C77700", label="constant model  f(q)        slope 1")
ax.loglog(hs_lin, err_lin, "o-", color="#0E7C7B", label="linear model  f(q) + J·δ   slope 2")
ax.set_xlabel("step length h (rad)"); ax.set_ylabel("error in hand position (m)")
ax.set_title("The Jacobian's error vanishes quadratically", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.show()

**Exercise 7.** A clinician operating the rehabilitation arm above wants the hand to move with velocity `xdot = np.array([0.05, -0.02])` m/s while the arm is at the configuration `q` used above (30 deg, 45 deg). Using the analytic Jacobian at that configuration, solve for the required joint angular velocities `qdot = J^-1 @ xdot` (inverse velocity kinematics), and verify `J @ qdot` reproduces the target `xdot`.

> 🤖 *Gemini tip:* "I have the Jacobian matrix of a 2-link robot arm and a desired hand velocity — show me how to solve for the joint velocities using inverse velocity kinematics, and how to verify the result."

In [ ]:
xdot = np.array([0.05, -0.02])

# Your code here

---
## 7 · The three losses you will actually differentiate

Almost every gradient in Géron's Chapters 4 and 9–11 is one of three, and all three end up in the
**same shape**. Write $z$ for the model's raw output (the *logit*), $\hat y$ for the prediction after
the activation, and $y$ for the target:

| task | activation | loss | $\partial L/\partial z$ |
|---|---|---|---|
| regression | none, $\hat y = z$ | $\tfrac12(\hat y - y)^2$ | $\hat y - y$ |
| binary | sigmoid | $-[\,y\log\hat y + (1-y)\log(1-\hat y)\,]$ | $\hat y - y$ |
| multi-class | softmax | $-\sum_c y_c \log \hat y_c$ | $\hat{\mathbf y} - \mathbf y$ |

The activation's derivative and the loss's derivative cancel each other exactly — that is *why*
these pairings are the standard ones, and why a framework can hand you `pred - target` as the
starting gradient of backpropagation. Never differentiate the sigmoid and the log separately: it is
both more work and numerically worse.


In [ ]:
sig = lambda z: 1/(1 + np.exp(-z))                     # the logistic sigmoid

def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))      # shift for numerical stability
    return e / e.sum(axis=-1, keepdims=True)

def central(f, x, h=1e-6):                             # scalar-in, scalar-out
    return (f(x + h) - f(x - h)) / (2*h)

# 1) regression: predicted VO2 vs measured
z_r, y_r = 41.0, 38.5
print("MSE      dL/dz  analytic", z_r - y_r,
      " numeric", round(central(lambda z: 0.5*(z - y_r)**2, z_r), 6))

# 2) binary: one breast-cancer sample, logit z, label y=1 (malignant)
z_b, y_b = 0.8, 1.0
bce = lambda z: -(y_b*np.log(sig(z)) + (1 - y_b)*np.log(1 - sig(z)))
print("log-loss dL/dz  analytic", round(sig(z_b) - y_b, 6),
      " numeric", round(central(bce, z_b), 6))

# 3) multi-class: an ECG beat scored as normal / PVC / atrial-fibrillation
z_m = np.array([1.2, -0.4, 0.3])
y_m = np.array([0.0, 1.0, 0.0])                        # the beat really is a PVC
ce  = lambda z: -(y_m*np.log(softmax(z))).sum()
ana = softmax(z_m) - y_m
num = np.array([central(lambda t, i=i: ce(z_m + t*np.eye(3)[i]), 0.0) for i in range(3)])
print("softmax  dL/dz  analytic", ana.round(6))
print("                numeric ", num.round(6))
print("predicted probabilities:", softmax(z_m).round(4), " -> loss", round(ce(z_m), 4))


**Exercise 8.** An EMG-based gesture classifier scores three gestures — rest, grasp, pinch — with
logits `z = np.array([0.5, 2.1, -0.3])`, and the true gesture is *grasp*, so `y = [0, 1, 0]`.
(a) Compute the softmax probabilities and the cross-entropy loss. (b) Compute `dL/dz` analytically as
`softmax(z) - y` and verify every component against a central finite difference. (c) Show that the
three components of `dL/dz` sum to zero, and explain in one line why they must.

> 🤖 *Gemini tip:* "Show me a numerically stable softmax in NumPy, the cross-entropy loss for a one-hot target, and a finite-difference check that the gradient of the loss with respect to the logits equals softmax(z) - y."


In [ ]:
z_g = np.array([0.5, 2.1, -0.3])
y_g = np.array([0.0, 1.0, 0.0])

# Your code here

---
## 8 · Backpropagation on a tiny graph, and one layer's gradients

Backprop is the chain rule applied backwards through a computational graph. For `L = (a*b + c)^2`
we sweep forward, then accumulate local derivatives backward — and then do the same for a whole
dense layer, which is where the two formulas you will meet in every framework come from.


In [ ]:
a, b, c = 2.0, 3.0, 1.0
# forward
u = a*b; v = u + c; Lval = v**2
# backward (each gradient = product of local derivatives along the path)
dL_dv = 2*v
dL_du = dL_dv * 1
dL_da = dL_du * b
dL_db = dL_du * a
dL_dc = dL_dv * 1
print(f"L={Lval} | dL/da={dL_da}, dL/db={dL_db}, dL/dc={dL_dc}")
# check dL/da numerically
Lf = lambda a_: (a_*b + c)**2
print("numeric dL/da:", central_diff(Lf, a))

**Exercise 9.** Backpropagate through a tiny BMI-like graph: `L = (w/h - target)**2`, with `w = 70.0` (kg), `h = 1.75` (m), `target = 22.0` (kg/m^2). Sweep forward to get `L`, then backward (by hand, through the division) to get `dL/dw` and `dL/dh`, and check both against the central difference.

> 🤖 *Gemini tip:* "Walk me through backpropagating by hand through a small computational graph that involves a division, like L = (w/h - target)^2, to get dL/dw and dL/dh, then show me how to check the result with a central-difference approximation."

In [ ]:
w, h, target = 70.0, 1.75, 22.0

# Your code here

### 8.1 · One layer's gradients: the two formulas you will meet everywhere

A dense layer computes $\mathbf z = \mathbf W\mathbf x + \mathbf b$. Suppose backpropagation has
already delivered $\boldsymbol\delta = \partial L/\partial \mathbf z$ (from Section 7, for a softmax
output that is just $\hat{\mathbf y} - \mathbf y$). Then

$$\frac{\partial L}{\partial \mathbf W} \;=\; \boldsymbol\delta\,\mathbf x^{\mathsf T},
\qquad
\frac{\partial L}{\partial \mathbf b} \;=\; \boldsymbol\delta,
\qquad
\frac{\partial L}{\partial \mathbf x} \;=\; \mathbf W^{\mathsf T}\boldsymbol\delta .$$

An **outer product** to update the weights, and a multiplication by $\mathbf W^{\mathsf T}$ to hand
the signal to the layer below. Check the shapes and you can never mis-write them: if $\mathbf W$ is
$n_\text{out}\times n_\text{in}$, then $\boldsymbol\delta\mathbf x^{\mathsf T}$ is
$(n_\text{out}\times 1)(1 \times n_\text{in})$, the shape of $\mathbf W$, and
$\mathbf W^{\mathsf T}\boldsymbol\delta$ is $n_\text{in}$ long, the shape of $\mathbf x$.


In [ ]:
# An 8-channel EMG feature vector -> 3 gestures, one layer, gradients checked numerically.
rng = np.random.default_rng(1)
n_in, n_out = 8, 3
x_e = rng.normal(size=n_in)
W_e = rng.normal(size=(n_out, n_in))*0.4
b_e = np.zeros(n_out)
y_e = np.array([0.0, 1.0, 0.0])                    # the gesture really is "grasp"

def loss_Wb(W, b):
    return -(y_e*np.log(softmax(W @ x_e + b))).sum()

delta = softmax(W_e @ x_e + b_e) - y_e             # dL/dz, from Section 7
dW = np.outer(delta, x_e)                          # dL/dW  = delta x^T
db = delta                                         # dL/db  = delta
dx = W_e.T @ delta                                 # dL/dx  = W^T delta
print("shapes:  W", W_e.shape, " delta", delta.shape, " dW", dW.shape, " dx", dx.shape)

# finite-difference every entry of W
h, dW_num = 1e-6, np.zeros_like(W_e)
for i in range(n_out):
    for j in range(n_in):
        E = np.zeros_like(W_e); E[i, j] = h
        dW_num[i, j] = (loss_Wb(W_e + E, b_e) - loss_Wb(W_e - E, b_e))/(2*h)
print("max |dW analytic - dW numeric| :", f"{np.abs(dW - dW_num).max():.2e}")

dx_num = np.array([( -(y_e*np.log(softmax(W_e @ (x_e + h*np.eye(n_in)[j]) + b_e))).sum()
                     -(-(y_e*np.log(softmax(W_e @ (x_e - h*np.eye(n_in)[j]) + b_e))).sum()) )/(2*h)
                   for j in range(n_in)])
print("max |dx analytic - dx numeric| :", f"{np.abs(dx - dx_num).max():.2e}")


**Exercise 10.** Stack a second layer on the gesture classifier: `h = np.tanh(W1 @ x + b1)` with
`W1` of shape `(5, 8)`, then `z = W2 @ h + b2` with `W2` of shape `(3, 5)`, and the same softmax
cross-entropy loss. Backpropagate by hand: start from `delta2 = softmax(z) - y`, get `dL/dW2` and
`dL/dh`, push through the `tanh` (whose derivative is `1 - np.tanh(...)**2`) to get `delta1`, and
finally `dL/dW1`. Verify `dL/dW1` against central finite differences.

> 🤖 *Gemini tip:* "Walk me through backpropagation for a two-layer network x -> tanh -> softmax with cross-entropy, writing each gradient as an outer product or a matrix-transpose product, and show me a NumPy finite-difference check for the first layer's weight gradient."


In [ ]:
rng2 = np.random.default_rng(7)
x2 = rng2.normal(size=8)
W1 = rng2.normal(size=(5, 8))*0.4;  b1 = np.zeros(5)
W2 = rng2.normal(size=(3, 5))*0.4;  b2 = np.zeros(3)
y2 = np.array([0.0, 0.0, 1.0])

# Your code here

---
## 9 · Why depth makes gradients vanish or explode

Backpropagation multiplies one factor per layer. Through $L$ layers the gradient reaching the first
layer carries a product of $L$ Jacobians, so its size behaves like $\rho^{L}$ where $\rho$ is a
typical per-layer factor. There is no middle ground: $\rho<1$ and the signal dies exponentially,
$\rho>1$ and it blows up. The sigmoid is the classic offender — $\sigma'(z) = \sigma(1-\sigma) \le
\tfrac14$, so **every** sigmoid layer shrinks the gradient by at least a factor of 4 even before the
weights are counted. This is the single practical reason ReLU and careful initialisation replaced
sigmoids in deep networks, and it is the background to Géron's Chapter 11.


In [ ]:
print("max of sigma'(z) =", round((sig(0)*(1 - sig(0))), 4), "at z = 0")
print("so 10 sigmoid layers shrink a gradient by at least", f"{0.25**10:.2e}", "on the derivative alone")

def backprop_norm(depth, scale, activation, seed=0):
    # Norm of dL/dx after pushing a unit gradient back through `depth` layers.
    r = np.random.default_rng(seed)
    n = 32
    x = r.normal(size=n)
    Ws, acts = [], []
    for _ in range(depth):                                   # forward
        W = r.normal(size=(n, n))*scale
        a = W @ x
        x = sig(a) if activation == "sigmoid" else np.maximum(a, 0.0)
        Ws.append(W); acts.append(a)
    g = r.normal(size=n); g /= np.linalg.norm(g)             # unit gradient at the top
    for W, a in zip(reversed(Ws), reversed(acts)):           # backward: a chain of vector-Jacobian products (9.1)
        g = g * (sig(a)*(1 - sig(a)) if activation == "sigmoid" else (a > 0).astype(float))
        g = W.T @ g
    return np.linalg.norm(g)

depths = np.arange(1, 26)
curves = {
    "sigmoid, scale 0.1":      [backprop_norm(d, 0.10, "sigmoid") for d in depths],
    "sigmoid, Xavier 1/sqrt(n)": [backprop_norm(d, 1/np.sqrt(32), "sigmoid") for d in depths],
    "sigmoid, scale 0.5":      [backprop_norm(d, 0.50, "sigmoid") for d in depths],
    "ReLU, He sqrt(2/n)":      [backprop_norm(d, np.sqrt(2/32), "relu") for d in depths],
}
for name, c in curves.items():
    print(f"{name:26s} depth 1: {c[0]:9.2e}   depth 25: {c[-1]:9.2e}")

fig, ax = plt.subplots(figsize=(6.6, 4))
for (name, c), col in zip(curves.items(), ["#B23A48", "#C77700", "#1B6CA8", "#0E7C7B"]):
    ax.semilogy(depths, c, "o-", ms=3, color=col, label=name)
ax.set_xlabel("number of layers"); ax.set_ylabel("|gradient| reaching layer 1")
ax.set_title("Gradient magnitude is exponential in depth")
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both"); plt.show()

**Exercise 11.** Take the `backprop_norm` helper above with the `"relu"` activation and a weight
scale you choose. Scan `scale` over `np.linspace(0.10, 0.35, 26)` at a fixed depth of 20, and find
the scale at which the gradient norm is closest to its starting value of 1 — i.e. the network neither
vanishes nor explodes. Compare your answer to He initialisation, `np.sqrt(2/32)`. Why should the
critical scale depend on the layer width `n = 32`?

> 🤖 *Gemini tip:* "Explain why the variance of W @ x grows with the number of inputs n, and how He initialisation scale sqrt(2/n) is chosen to keep the signal variance constant through a ReLU layer."


In [ ]:
scales = np.linspace(0.10, 0.35, 26)

# Your code here

### 9.1 · Backprop is a product of Jacobians — and why it runs backwards

Each layer $\mathbf h_k=\tanh(\mathbf W_k\mathbf h_{k-1})$ has a Jacobian
$\mathbf J_k=\partial\mathbf h_k/\partial\mathbf h_{k-1}=\operatorname{diag}\!\big(1-\mathbf h_k^2\big)\,\mathbf W_k$,
and the chain rule for a composition is the **product** of those Jacobians. For a scalar loss on top,

$$\nabla_{\mathbf x}L^{\mathsf T} \;=\; \mathbf g^{\mathsf T}\,\mathbf J_3\,\mathbf J_2\,\mathbf J_1,
\qquad \mathbf g^{\mathsf T}=\partial L/\partial\mathbf h_3\ \ (\text{a single row}).$$

That is Notebook 1's *composition is matrix multiplication* seen through calculus. Matrix products are
**associative**, so there are two ways to evaluate the chain and they give the same answer:

* **from the loss end** — $\big((\mathbf g^{\mathsf T}\mathbf J_3)\mathbf J_2\big)\mathbf J_1$: a row vector
  times a matrix, three times. Each step costs $n^2$. This is **reverse mode**, i.e. backprop.
* **from the input end** — $\mathbf g^{\mathsf T}\big(\mathbf J_3(\mathbf J_2\mathbf J_1)\big)$: full
  $n\times n$ matrix products. Each costs $n^3$. This is **forward mode**.

Same number, roughly $n$ times the work. The PDF (Module 3, §5.3) makes the argument; here we check it.

In [ ]:
# A 3-layer tanh network: build every layer's Jacobian explicitly, then multiply the chain both ways.
jc_rng = np.random.default_rng(3)
jc_n = 6
jc_Ws = [jc_rng.normal(size=(jc_n, jc_n)) / np.sqrt(jc_n) for _ in range(3)]
jc_x, jc_t = jc_rng.normal(size=jc_n), jc_rng.normal(size=jc_n)


def jc_forward(x):
    hs, h = [], x
    for W in jc_Ws:
        h = np.tanh(W @ h)
        hs.append(h)
    return hs


def jc_loss(x):
    return 0.5 * np.sum((jc_forward(x)[-1] - jc_t) ** 2)


jc_hs = jc_forward(jc_x)
jc_Js = [np.diag(1 - h**2) @ W for W, h in zip(jc_Ws, jc_hs)]     # J_k = diag(tanh') W_k
jc_g = jc_hs[-1] - jc_t                                              # dL/dh_3

grad_reverse = ((jc_g @ jc_Js[2]) @ jc_Js[1]) @ jc_Js[0]             # thin end first
grad_forward = jc_g @ (jc_Js[2] @ (jc_Js[1] @ jc_Js[0]))             # matrices first
grad_numeric = np.array([(jc_loss(jc_x + 1e-6*e) - jc_loss(jc_x - 1e-6*e)) / 2e-6
                         for e in np.eye(jc_n)])
print("reverse order :", grad_reverse.round(6))
print("forward order :", grad_forward.round(6))
print("finite diff   :", grad_numeric.round(6))
print("all three agree:", np.allclose(grad_reverse, grad_forward) and np.allclose(grad_reverse, grad_numeric, atol=1e-7))

# The backward loop inside backprop_norm (Section 9) is exactly this chain, one layer at a time:
# multiplying the row g^T by J_k = diag(s') W_k is the same as  W_k^T @ (s' * g).
g_loop = jc_g.copy()
for W, h in zip(reversed(jc_Ws), reversed(jc_hs)):
    g_loop = W.T @ ((1 - h**2) * g_loop)
print("backprop_norm-style loop gives the same gradient:", np.allclose(g_loop, grad_reverse))

In [ ]:
import time

# The cost of each order: an (a x b)(b x c) product takes a*b*c multiply-adds.
def cost_reverse(n, L): return L * n * n                        # L row-times-matrix products
def cost_forward(n, L): return (L - 1) * n**3 + n * n            # L-1 matrix products, then one row

print("multiply-adds for a chain of L layers of width n")
for n, L in [(32, 25), (512, 10), (4096, 50)]:
    cr, cf = cost_reverse(n, L), cost_forward(n, L)
    print(f"  n={n:5d}, L={L:2d}:  reverse {cr:.2e} | forward {cf:.2e} | forward/reverse = {cf/cr:,.0f}x")

# Wall-clock at realistic widths. Optimised matrix-multiply libraries run big matrix products far more
# efficiently than vector products, so the measured gap is smaller than the operation count - but it
# is still large, and it keeps growing with width.
t_rng = np.random.default_rng(0)
for n in (256, 512):
    Wb = [t_rng.normal(size=(n, n)) / np.sqrt(n) for _ in range(8)]
    gv = t_rng.normal(size=n)
    t0 = time.perf_counter(); v = gv.copy()
    for W in reversed(Wb):
        v = v @ W
    t_rev = time.perf_counter() - t0
    t0 = time.perf_counter(); M = Wb[0].copy()
    for W in Wb[1:]:
        M = W @ M
    v2 = gv @ M
    t_fwd = time.perf_counter() - t0
    print(f"  measured, n={n}, 8 layers: reverse {1e3*t_rev:6.2f} ms | forward {1e3*t_fwd:7.2f} ms"
          f" | same answer: {np.allclose(v, v2, rtol=1e-6, atol=1e-8)}")

print("The measured gap is far smaller than the multiply-add ratio (~220x at n=256): optimised BLAS\n"
      "runs matrix-matrix products much faster per flop than vector-matrix ones. The count, not the clock,\n"
      "is what grows without bound - at n = 4096 no BLAS trick closes a factor of 4,000.")

widths = np.logspace(1, 4, 30)
fig, ax = plt.subplots(figsize=(6.0, 3.8))
ax.loglog(widths, cost_forward(widths, 10), color="#B23A48", lw=2, label="forward mode  ~ n³")
ax.loglog(widths, cost_reverse(widths, 10), color="#0E7C7B", lw=2, label="reverse mode (backprop)  ~ n²")
ax.set_xlabel("layer width n"); ax.set_ylabel("multiply-adds, 10 layers")
ax.set_title("Same gradient, two orders of multiplication", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3, which="both")
plt.show()

**The rule behind both modes: start multiplying from the thin end of the chain.** Training has a *scalar*
loss, so the chain begins with a single row $\mathbf g^{\mathsf T}$ on the output side — start there, and
every product stays a vector. That is why backprop runs backwards, and why PyTorch's `.backward()` is
called on the loss. But the rule is about shape, not direction: when a model has *one* input and *many*
outputs, the thin end is on the input side and forward mode wins instead. The next exercise is exactly
that case.

**Exercise 12.** A pharmacokinetic model passes a single dosing parameter through three stages to predicted plasma
concentrations at $m=200$ sampling times. The stage Jacobians have shapes $\mathbf J_1$: $50\times1$,
$\mathbf J_2$: $50\times50$ and $\mathbf J_3$: $200\times50$, so the sensitivity of every concentration to
the dose is $\mathbf J_3\mathbf J_2\mathbf J_1$ ($200\times1$).
(a) Build random matrices of those shapes and compute the product in both orders, confirming they agree.
(b) Count the multiply-adds each order needs, using $a\cdot b\cdot c$ for an $(a\times b)(b\times c)$ product.
(c) Which order wins here? State the single rule that covers both this case and training a network.

> 🤖 *Gemini tip:* "Explain forward-mode versus reverse-mode automatic differentiation in terms of the order you multiply a chain of Jacobian matrices, and when each one is cheaper."

In [ ]:
pk_rng = np.random.default_rng(11)

# Your code here

---
## 10 · Gradient descent from scratch

Minimise a loss by stepping downhill: `w <- w - eta * grad`. We fit a logistic-regression decision
boundary to two breast-cancer features by minimising the cross-entropy loss with hand-written
gradients, and watch the loss fall. The step direction is the steepest one by Section 3, and the
step itself minimises the first-order Taylor model of Section 5.


In [ ]:
from sklearn.datasets import load_breast_cancer
bc = load_breast_cancer()
# two standardised features + bias; labels 0/1
X = bc.data[:, [0, 3]]                       # mean radius, mean area
X = (X - X.mean(0)) / X.std(0)
X = np.c_[np.ones(len(X)), X]                # add bias column
y = bc.target.astype(float)

def sigmoid(z): return 1/(1+np.exp(-z))
def loss(w):
    p = sigmoid(X @ w)
    return -np.mean(y*np.log(p+1e-9) + (1-y)*np.log(1-p+1e-9))
def grad(w):
    p = sigmoid(X @ w)
    return X.T @ (p - y) / len(y)            # tidy logistic-regression gradient

w = np.zeros(3); eta = 0.5; history = []
for step in range(300):
    w -= eta * grad(w)
    history.append(loss(w))
acc = np.mean((sigmoid(X @ w) > 0.5) == y)
print("final loss:", round(history[-1], 4), "| training accuracy:", round(acc, 3))
plt.figure(figsize=(6,3)); plt.plot(history)
plt.xlabel("step"); plt.ylabel("cross-entropy loss"); plt.title("Gradient descent")
plt.grid(alpha=0.3); plt.show()

### 10.1 · The learning-rate law: reading $\eta^*$ off the Hessian

Guessing learning rates is unnecessary. Near a minimum, Taylor (Section 5) says the loss is a bowl
$L\approx L^*+\tfrac12\,\mathbf e^{\mathsf T}\mathbf H\,\mathbf e$ with $\mathbf e=\mathbf w-\mathbf w^*$, so one
descent step gives $\mathbf e\leftarrow(\mathbf I-\eta\mathbf H)\,\mathbf e$.

Now use the **conjugation trick** from Notebook 1 (§13.2). In the eigenbasis of $\mathbf H$, the matrix
$\mathbf I-\eta\mathbf H$ is diagonal, so descent splits into independent one-variable updates, one per
eigen-direction:
$$c_i\;\leftarrow\;(1-\eta\lambda_i)\,c_i .$$
Each shrinks only if $|1-\eta\lambda_i|<1$, i.e. $0<\eta<2/\lambda_i$, and the tightest of those is the
largest eigenvalue:

$$\boxed{\;\eta^*=\frac{2}{\lambda_{\max}}\;}$$

Above it, the stiffest mode *grows* by $|1-\eta\lambda_{\max}|>1$ every step. Below it, every mode shrinks
— but the slowest shrinks only by $\max_i|1-\eta\lambda_i|$ per step — at the best fixed step that is $(\kappa-1)/(\kappa+1)\approx1-2/\kappa$, with $\kappa=\lambda_{\max}/\lambda_{\min}$ the condition number.
The PDF states this in §6.1; here we test it on the VO₂ data from Section 5.

In [ ]:
# The VO2 least-squares problem from Section 5 (rebuilt with fresh names). Its loss is exactly quadratic,
# so H is constant and the law should be sharp.
rng_v = np.random.default_rng(0)
m_v = 60
X_v = np.c_[np.ones(m_v), rng_v.normal(size=m_v), rng_v.normal(size=m_v)]
y_v = X_v @ np.array([8.0, 2.5, -1.2]) + rng_v.normal(scale=0.4, size=m_v)
g_v = lambda w: X_v.T @ (X_v @ w - y_v)
H_v = X_v.T @ X_v
lam_v, V_v = np.linalg.eigh(H_v)
w_opt_v = np.linalg.lstsq(X_v, y_v, rcond=None)[0]

eta_star_v = 2 / lam_v.max()
print("eigenvalues of H:", lam_v.round(2), f"-> kappa = {lam_v.max()/lam_v.min():.2f}")
print(f"PREDICTED threshold  eta* = 2/lambda_max = {eta_star_v:.5f}")


def descend_v(eta, steps):
    w, errs, modes = np.zeros(3), [], []
    for _ in range(steps):
        modes.append(V_v.T @ (w - w_opt_v))              # the error, in H's eigenbasis
        errs.append(np.linalg.norm(w - w_opt_v))
        w = w - eta * g_v(w)
    return np.array(errs), np.abs(np.array(modes))


err_lo, _ = descend_v(0.98 * eta_star_v, 300)
err_hi, _ = descend_v(1.02 * eta_star_v, 300)
print(f"TESTED  eta = 0.98 eta*: error after 300 steps {err_lo[-1]:.2e}   (converges)")
print(f"        eta = 1.02 eta*: error after 300 steps {err_hi[-1]:.2e}   (diverges)")

# Decoupling: each eigen-mode should decay on its own, exactly as (1 - eta*lambda_i)^k.
eta_d, K = 0.9 * eta_star_v, 18                          # 18 steps keeps every mode above float64's floor
_, modes = descend_v(eta_d, K)
pred = np.array([modes[0] * np.abs(1 - eta_d * lam_v) ** k for k in range(K)])
print("each mode decays exactly as (1 - eta*lambda)^k:", np.allclose(modes, pred, rtol=1e-6, atol=1e-14))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.9))
ax[0].semilogy(err_lo, color="#0E7C7B", label="η = 0.98 η*   converges")
ax[0].semilogy(err_hi, color="#B23A48", label="η = 1.02 η*   diverges")
ax[0].set_xlabel("step"); ax[0].set_ylabel("distance to the optimum")
ax[0].set_title("A 4% change in η, on either side of 2/λmax", fontsize=10)
ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
for i, col in enumerate(["#1B6CA8", "#C77700", "#B23A48"]):
    f = 1 - eta_d * lam_v[i]
    ax[1].semilogy(modes[:, i], "o", ms=4, color=col)
    ax[1].semilogy(pred[:, i], "-", lw=1, color=col, label=f"λ = {lam_v[i]:.1f}: factor {f:+.3f} per step")
ax[1].set_xlabel("step"); ax[1].set_ylabel("|error along eigenvector|")
ax[1].set_title("In H's eigenbasis each mode decays alone", fontsize=10)
ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()
print("Note the signs: near eta*, the STIFFEST mode has a factor close to -1, so it flips sign every step")
print("and converges slowest of all. That mode is the one that sets the limit.")

**The ravine.** The VO₂ problem is well conditioned ($\kappa\approx1.5$), so every mode converges at a
similar pace. The breast-cancer data of this section is not. *Mean radius* and *mean area* are almost the
same measurement — area grows with radius squared — so the loss barely changes if you shift weight from
one to the other. That direction is nearly flat, and the contours become a long, narrow ravine.

To draw it honestly we hold the intercept at its optimum and descend on the two feature weights alone, so
the path and the contours describe the same function.

In [ ]:
# The optimum of the full logistic loss, found by Newton's method (Section 5) - a handful of steps.
def hess_logistic(w, A=X):
    p = sigmoid(A @ w)
    return (A * (p * (1 - p))[:, None]).T @ A / len(y)


w_bc_opt = np.zeros(3)
for _ in range(12):
    w_bc_opt = w_bc_opt - np.linalg.solve(hess_logistic(w_bc_opt), grad(w_bc_opt))
print("corr(mean radius, mean area) =", round(np.corrcoef(bc.data[:, 0], bc.data[:, 3])[0, 1], 4))

# Hold the intercept fixed; the two feature weights form a genuine 2-parameter problem.
b_fix, X_feat = w_bc_opt[0], X[:, 1:]
loss_feat = lambda v: loss(np.r_[b_fix, v])
grad_feat = lambda v: grad(np.r_[b_fix, v])[1:]
H_feat = hess_logistic(np.r_[b_fix, w_bc_opt[1:]])[1:, 1:]
lam_f, V_f = np.linalg.eigh(H_feat)
eta_star_f = 2 / lam_f.max()
print(f"feature Hessian eigenvalues {lam_f.round(5)} -> kappa = {lam_f.max()/lam_f.min():.0f}, eta* = {eta_star_f:.1f}")

v_opt = w_bc_opt[1:]
start = v_opt + 1.2 * V_f[:, 1] + 5.5 * V_f[:, 0]            # off-optimum along both directions
A1 = np.linspace(v_opt[0] - 7, v_opt[0] + 7, 150)
A2 = np.linspace(v_opt[1] - 7, v_opt[1] + 7, 150)
Z = np.array([[loss_feat(np.array([a1, a2])) for a1 in A1] for a2 in A2])

fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.6))
for ax, frac in [(axs[0], 0.35), (axs[1], 0.93)]:
    v, path = start.copy(), [start.copy()]
    for _ in range(40):
        v = v - frac * eta_star_f * grad_feat(v)
        path.append(v.copy())
    path = np.array(path)
    ax.contour(A1, A2, Z, levels=np.quantile(Z, np.linspace(0.004, 0.8, 18)), colors="teal", linewidths=0.7)
    ax.plot(path[:, 0], path[:, 1], "-o", ms=2.5, lw=0.9, color="#B23A48")
    ax.plot(*start, "s", color="k", ms=5); ax.plot(*v_opt, "*", ms=13, color="#C77700")
    left_flat = abs((path[-1] - v_opt) @ V_f[:, 0]) / 5.5
    left_stiff = abs((path[-1] - v_opt) @ V_f[:, 1]) / 1.2
    ax.set_title(f"η = {frac:.2f} η*:  after 40 steps {100*left_flat:.0f}% of the along-ravine error remains",
                 fontsize=9)
    ax.set_xlabel("weight on mean radius"); ax.set_ylabel("weight on mean area")
    print(f"eta = {frac:.2f} eta*: across-ravine error left {100*left_stiff:.1f}% | along-ravine left {100*left_flat:.0f}%")
plt.tight_layout(); plt.show()
print("The steep direction is gone within a few steps; the flat one crawls. That is kappa at work.")

**Exercise 13.** Now use the law instead of guessing, on the breast-cancer model of this section. Its Hessian is
$\mathbf H(\mathbf w)=\mathbf X^{\mathsf T}\operatorname{diag}\big(p(1-p)\big)\mathbf X/m$ with $p=\sigma(\mathbf X\mathbf w)$
— the `hess_logistic` helper above computes it.
(a) **Predict.** Compute $\mathbf H$ at the starting point $\mathbf w=\mathbf 0$, its largest eigenvalue, and
hence $\eta^*=2/\lambda_{\max}$.
(b) **Test.** Run 300 steps of descent at $\eta=1.05\,\eta^*$. Does it diverge, as the law seems to promise?
(c) **Explain.** Compute $\mathbf H$ at the trained optimum `w_bc_opt` and read off a new $\eta^*$. Why has it
changed so much? *(Think about $p(1-p)$ once the model is confident.)*
(d) **Confirm.** Scan $\eta$ upwards and find where the loss first stops decreasing monotonically. Which of
your two predictions does it match?

> 🤖 *Gemini tip:* "For logistic regression, how does the Hessian change as training proceeds, and how does that change the largest learning rate gradient descent can use without diverging?"

In [ ]:
# Your code here

---
## 11 · The same gradient three ways (autograd)

PyTorch's **autograd** records operations and returns exact gradients via reverse-mode autodiff (the
backprop of Section 8, automated). We compute one gradient by hand, by finite difference, and by
autograd — they agree to machine precision — then run a 3-line training loop with autograd.


In [ ]:
try:
    import torch
    HAVE_TORCH = True
    print("PyTorch", torch.__version__)
except ImportError:
    HAVE_TORCH = False
    print("torch not installed here; this section runs in Colab (pre-installed).")

In [ ]:
if HAVE_TORCH:
    # gradient of L(w) = 0.5*(w*x - y)^2 at w=2, with x=3, y=7
    w_t = torch.tensor(2.0, requires_grad=True)
    L_t = 0.5*(w_t*3.0 - 7.0)**2
    L_t.backward()                       # autograd fills w_t.grad
    print("by hand      :", (2.0*3.0 - 7.0)*3.0)
    print("finite diff  :", central_diff(lambda w: 0.5*(w*3-7)**2, 2.0))
    print("autograd     :", w_t.grad.item())

In [ ]:
if HAVE_TORCH:
    # A 3-line training loop with autograd: minimise (w*3 - 7)^2
    w_t = torch.tensor(0.0, requires_grad=True)
    opt = torch.optim.SGD([w_t], lr=0.05)
    for _ in range(100):
        opt.zero_grad()
        L_t = (w_t*3.0 - 7.0)**2
        L_t.backward()                   # compute gradient
        opt.step()                       # w <- w - lr*grad
    print("converged w:", round(w_t.item(), 4), "(target 7/3 =", round(7/3, 4), ")")

**Exercise 14.** Using PyTorch autograd, fit the elimination rate `k` in `C(t) = C0 * exp(-k*t)` to noisy concentration data generated from `C0=50.0, k_true=0.35`. Start from a guess `k=0.1`, run gradient descent for 300 steps (a small learning rate keeps it stable) minimising the mean squared error, and report the fitted `k`.

> 🤖 *Gemini tip:* "I have noisy exponential-decay data in PyTorch tensors and want to fit the decay rate with autograd — show me how to set up a trainable parameter, a loss function, and a training loop with SGD."

In [ ]:
if HAVE_TORCH:
    t_data = torch.linspace(0, 8, 20)
    C0_true, k_true = 50.0, 0.35
    torch.manual_seed(0)
    noise = torch.randn(20) * 1.0
    C_data = C0_true*torch.exp(-k_true*t_data) + noise

# Your code here

---
### You now have the differential half
Derivatives and the chain rule, numerical derivatives and gradient checking, the gradient with its
tangent plane and the total differential along a path, Taylor as the local model every optimiser
minimises, the Jacobian, the three losses and the single gradient shape they share, backprop with one
layer's outer-product gradients and as a product of Jacobians multiplied from the thin end, why depth
makes gradients vanish or explode, gradient descent from scratch with its learning rate predicted from
the Hessian's eigenvalues, and PyTorch autograd — the calculus behind Géron's Chapters 4 and 9–11.
**Next:** Notebook 2 turns to integration and the PID controller.